In [ ]:
# ================================
# 04-hyperparameter-search (corrected)
# Single LR per method, pooled across both languages
# Classifier head trainable — this was the root bug fix
# ================================

!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ================================
# Imports
# ================================
import os, time, json
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
print(os.listdir(DATA_ROOT))
print(os.listdir(f"{DATA_ROOT}/hi"))  # confirm train_2000 exists

In [ ]:
# ================================
# Search configuration
# ================================
SWEEP_BUDGET = 2000          # representative mid-size budget, not the full grid
LANGUAGES = ["hi", "te"]     # pooled across both — LR choice must not depend on language
LEARNING_RATES = [5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3]
EPOCHS = 10
BATCH_SIZE = 32
SEEDS = [42, 123]            # 2 seeds per config, to reduce noise in LR selection
METHODS = ["lora", "dora", "ia3"]

In [ ]:
# ================================
# Data loader
# ================================
def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"],
                          truncation=True, padding="max_length", max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")
    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch"); valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=64)
    return train_loader, valid_loader

In [ ]:
# ================================
# Model builder — classifier trainable (the actual fix)
# ================================
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()

def build_model(method):
    base = load_base_model()
    if method == "lora":
        config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
                             task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],
                             modules_to_save=["classifier"])
        return get_peft_model(base, config)
    if method == "dora":
        config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none", use_dora=True,
                             task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],
                             modules_to_save=["classifier"])
        return get_peft_model(base, config)
    if method == "ia3":
        config = IA3Config(task_type=TaskType.SEQ_CLS,
                            target_modules=["key", "value", "output.dense"],
                            feedforward_modules=["output.dense"],
                            modules_to_save=["classifier"])
        return get_peft_model(base, config)
    raise ValueError(method)

# Verify the fix before running anything expensive
_test = build_model("lora")
assert any("classifier" in n for n, p in _test.named_parameters() if p.requires_grad), \
    "STOP — classifier still frozen"
print("✓ Classifier trainable")
del _test
torch.cuda.empty_cache()

In [ ]:
# ================================
# Train + evaluate one (method, language, lr, seed) config
# ================================
def train_and_evaluate(method, language, lr, seed, epochs=EPOCHS):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)

    train_loader, valid_loader = build_loaders(language, SWEEP_BUDGET)
    model = build_model(method)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average="macro")

    del model
    torch.cuda.empty_cache()
    return acc, f1

In [ ]:
# ================================
# Run search: 3 methods × 7 LRs × 2 langs × 2 seeds = 84 runs
# ================================
results = []
total_configs = len(METHODS) * len(LEARNING_RATES) * len(LANGUAGES) * len(SEEDS)
run_count = 0

for method in METHODS:
    for lr in LEARNING_RATES:
        for lang in LANGUAGES:
            for seed in SEEDS:
                run_count += 1
                print(f"\n[{run_count}/{total_configs}] {method} | lr={lr} | {lang} | seed={seed}")
                acc, f1 = train_and_evaluate(method, lang, lr, seed)
                results.append({
                    "method": method, "lr": lr, "language": lang, "seed": seed,
                    "accuracy": acc, "macro_f1": f1
                })
                print(f"  acc={acc:.4f}, f1={f1:.4f}")

sweep_df = pd.DataFrame(results)
sweep_df.to_csv("/kaggle/working/lr_sweep_raw_results.csv", index=False)
display(sweep_df)

In [ ]:
# ================================
# Aggregate by (method, lr), POOLED across both languages
# This is the key fix: one LR per method, not one per (method, language)
# ================================
summary = sweep_df.groupby(["method", "lr"]).agg({
    "accuracy": ["mean", "std"],
    "macro_f1": ["mean", "std"]
}).reset_index()
summary.columns = ["method", "lr", "acc_mean", "acc_std", "f1_mean", "f1_std"]
summary = summary.sort_values(["method", "f1_mean"], ascending=[True, False])
display(summary)

In [ ]:
# ================================
# Pick best LR per method — pooled across hi + te
# Hard assert: don't silently accept a "best" that's still at random chance
# ================================
best_lr_per_method = {}

for method in METHODS:
    sub = summary[summary["method"] == method].sort_values("f1_mean", ascending=False)
    best_row = sub.iloc[0]
    best_lr = best_row["lr"]
    best_acc = best_row["acc_mean"]
    best_f1 = best_row["f1_mean"]

    assert best_acc > 0.35, (
        f"STOP — {method}: best LR ({best_lr}) still at/near random chance "
        f"(acc={best_acc:.4f}). Expand the LR search range before proceeding."
    )

    best_lr_per_method[method] = float(best_lr)
    print(f"{method}: best LR = {best_lr} (mean acc={best_acc:.4f}, mean f1={best_f1:.4f})")

with open("/kaggle/working/chosen_lrs.json", "w") as f:
    json.dump(best_lr_per_method, f, indent=2)

print("\n✓ Saved chosen_lrs.json:", best_lr_per_method)